## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [23]:
import sys
print(sys.executable)

/Users/mjhalili/Desktop/EADA_Files/ML_Operations/mlops-and-systems-design/session_3/venv-session3/bin/python


In [24]:
# import packages
import pandas as pd
import joblib
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# Load the dataset
df = pd.read_csv("Churn_Modelling_train_test.csv")

# dataset analysis
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,4784,15729224,Jennings,710,France,Female,37.0,5,0.00,2,1.0,0.0,115403.31,0
1,1497,15799156,Okwuadigbo,569,Spain,Male,38.0,8,0.00,2,0.0,0.0,79618.79,0
2,1958,15674922,Beavers,710,France,Male,54.0,6,171137.62,1,1.0,1.0,167023.95,1
3,9174,15653572,Thornton,673,Spain,Male,43.0,8,127132.96,1,0.0,1.0,6009.27,1
4,9748,15775761,Iweobiegbunam,610,Germany,Female,69.0,5,86038.21,3,0.0,0.0,192743.06,1


In [25]:
# Churn Dataset Cleaning and Correlation-Ready Pipeline
# - Clean dataset for machine learning and analytics
# - Prepare correlation-ready numerical dataset
# - Handle missing values, duplicates, outliers, encoding
# - Generate engineered features
# - Export cleaned datasets
# Strip spaces from column names immediately
df.columns = df.columns.str.strip()
# Remove duplicates
df = df.drop_duplicates()
# Standardize datatypes
# Convert binary columns
binary_cols = ['HasCrCard', 'IsActiveMember', 'Exited']

for col in binary_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Missing values:
# - Numerical -> median
# - Categorical -> most frequent
numerical_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Numerical imputation
num_imputer = SimpleImputer(strategy='median')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Categorical imputation
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Clean string val
for col in categorical_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

# IQR for clipping outliers
numeric_features = [
    col for col in numerical_cols
    if col != 'Exited'
]

for col in numeric_features:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower_bound, upper_bound)


In [26]:
# Create engineered variables in preparation for correlation analysis
# Balance-to-Salary Ratio
if {'Balance', 'EstimatedSalary'}.issubset(df.columns):
    df['BalanceSalaryRatio'] = (
        df['Balance'] / (df['EstimatedSalary'] + 1)
    )

# Age Groups
if 'Age' in df.columns:
    df['AgeGroup'] = pd.cut(
        df['Age'],
        bins=[0, 25, 35, 50, 65, 100],
        labels=['18-25', '26-35', '36-50', '51-65', '65+']
    )

# Product Engagement Score
if {'NumOfProducts', 'IsActiveMember'}.issubset(df.columns):
    df['EngagementScore'] = (
        df['NumOfProducts'] * df['IsActiveMember']
    )

# Copy before encoding
clean_df = df.copy()

# One-hot encoding for nominal categories
encoded_df = pd.get_dummies(
    clean_df,
    columns=['Geography', 'Gender', 'AgeGroup'],
    drop_first=True
)

# Try correlation
correlation_matrix = encoded_df.corr(numeric_only=True)

# Sort by correlation with target variable
if 'Exited' in correlation_matrix.columns:

    churn_corr = (
        correlation_matrix['Exited']
        .sort_values(ascending=False)
    )

    print('\n================ CORRELATION WITH CHURN ================')
    print(churn_corr)


# Export the clean dataset
# Main  dataset
encoded_df.to_csv(
    'cleaned_churn_dataset.csv',
    index=False
)

# Correlation matrix
correlation_matrix.to_csv(
    'churn_correlation_matrix.csv'
)

# Descriptive analysis, Full statistical summary
stats_summary = encoded_df.describe(include='all').transpose()
print(stats_summary)

# Additional advanced metrics
advanced_stats = pd.DataFrame({
    'mean': encoded_df.mean(numeric_only=True),
    'median': encoded_df.median(numeric_only=True),
    'std_dev': encoded_df.std(numeric_only=True),
    'variance': encoded_df.var(numeric_only=True),
    'min': encoded_df.min(numeric_only=True),
    'max': encoded_df.max(numeric_only=True),
    'skewness': encoded_df.skew(numeric_only=True),
    'kurtosis': encoded_df.kurtosis(numeric_only=True)
})


================ CORRELATION WITH CHURN ================
Exited                1.000000
Age                   0.309255
AgeGroup_51-65        0.223137
Geography_Germany     0.175439
Balance               0.116550
AgeGroup_36-50        0.103516
BalanceSalaryRatio    0.027448
EstimatedSalary       0.010752
CustomerId           -0.006444
HasCrCard            -0.010679
Tenure               -0.013248
RowNumber            -0.019545
CreditScore          -0.024016
Geography_Spain      -0.053332
NumOfProducts        -0.063161
Gender_Male          -0.107566
EngagementScore      -0.145120
IsActiveMember       -0.159237
AgeGroup_26-35       -0.223402
AgeGroup_65+               NaN
Name: Exited, dtype: float64
                     count unique    top  freq             mean           std  \
RowNumber           8999.0    NaN    NaN   NaN       5002.25614   2884.146186   
CustomerId          8999.0    NaN    NaN   NaN  15690732.526058   71798.44394   
Surname               8999   2771  Smith    29    

In [5]:
# review categorical variables
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Categorical Variables:")
print(categorical_cols)

# Display unique values and counts
for col in categorical_cols:
    print(f"\n================ {col.upper()} ================")
    print(df[col].value_counts(dropna=False))

Categorical Variables:
['Surname', 'Geography', 'Gender']

================ SURNAME ================
Surname
Smith          29
Martin         25
Yeh            25
Walker         23
Shih           23
               ..
Beit            1
Onwuamaegbu     1
Edith           1
Holder          1
Macrossan       1
Name: count, Length: 2771, dtype: int64

================ GEOGRAPHY ================
Geography
France     4505
Spain      2250
Germany    2244
Name: count, dtype: int64

================ GENDER ================
Gender
Male      4905
Female    4094
Name: count, dtype: int64


In [27]:
# review binary variables

# Identify binary columns
binary_cols = []

for col in df.columns:
    if df[col].nunique() == 2:
        binary_cols.append(col)

print("Binary Variables:")
print(binary_cols)

# Display binary distributions
for col in binary_cols:
    print(f"\n================ {col.upper()} ================")
    print(df[col].value_counts(dropna=False))

Binary Variables:
['Gender', 'HasCrCard', 'IsActiveMember', 'Exited']

================ GENDER ================
Gender
Male      4905
Female    4094
Name: count, dtype: int64

================ HASCRCARD ================
HasCrCard
1.0    6350
0.0    2649
Name: count, dtype: int64

================ ISACTIVEMEMBER ================
IsActiveMember
1.0    4633
0.0    4366
Name: count, dtype: int64

================ EXITED ================
Exited
0.0    7160
1.0    1839
Name: count, dtype: int64


In [28]:
# review numerical variables

# Identify numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Numerical Variables:")
print(numerical_cols)

# Display descriptive statistics
print("\n================ NUMERICAL SUMMARY ================")
print(df[numerical_cols].describe().transpose())

# Additional statistics
for col in numerical_cols:
    print(f"\n================ {col.upper()} ================")
    print(f"Mean: {df[col].mean()}")
    print(f"Median: {df[col].median()}")
    print(f"Standard Deviation: {df[col].std()}")
    print(f"Variance: {df[col].var()}")
    print(f"Minimum: {df[col].min()}")
    print(f"Maximum: {df[col].max()}")
    print(f"Skewness: {df[col].skew()}")
    print(f"Kurtosis: {df[col].kurtosis()}")

Numerical Variables:
['RowNumber', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'BalanceSalaryRatio', 'EngagementScore']

================ NUMERICAL SUMMARY ================
                     count          mean           std          min  \
RowNumber           8999.0  5.002256e+03   2884.146186         2.00   
CustomerId          8999.0  1.569073e+07  71798.443940  15565701.00   
CreditScore         8999.0  6.506811e+02     96.446311       383.00   
Age                 8999.0  3.864619e+01      9.722342        18.00   
Tenure              8999.0  5.008223e+00      2.894250         0.00   
Balance             8999.0  7.621635e+04  62436.547243         0.00   
NumOfProducts       8999.0  1.528725e+00      0.568670         1.00   
HasCrCard           8999.0  7.056340e-01      0.455783         0.00   
IsActiveMember      8999.0  5.148350e-01      0.499808         0.00   
EstimatedSalary     8999.0 

In [29]:
# import packages, general ones
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

# Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [30]:
# Load the dataset
df = pd.read_csv("cleaned_churn_dataset.csv")

In [31]:
# Features
X = df.drop('Exited', axis=1)

# Labels
y = df['Exited']

print("Features Shape:", X.shape)
print("Labels Shape:", y.shape)

Features Shape: (8999, 20)
Labels Shape: (8999,)


In [32]:
# initialize scaler globally
scaler = StandardScaler()

# Preprocess the features - create a function or a class for it 
def transform(df: pd.DataFrame) -> pd.DataFrame:

    # create copy
    df = df.copy()

    # ensure all columns are numeric
    df = df.apply(pd.to_numeric, errors='coerce')

    # fill remaining unexpected nulls
    df = df.fillna(0)

    # scale features
    scaled_array = scaler.fit_transform(df)

    # convert back to dataframe
    df_scaled = pd.DataFrame(
        scaled_array,
        columns=df.columns,
        index=df.index
    )

    return df_scaled

In [33]:
# apply the transform function
X_processed = transform(X)

print(X_processed.head())

   RowNumber  CustomerId  Surname  CreditScore       Age    Tenure   Balance  \
0  -0.075679    0.536134      0.0     0.615080 -0.169329 -0.002841 -1.220769   
1  -1.215421    1.510193      0.0    -0.846954 -0.066468  1.033754 -1.220769   
2  -1.055572   -0.220219      0.0     0.615080  1.579318  0.342691  1.520368   
3   1.446520   -0.517596      0.0     0.231426  0.447840  1.033754  0.815539   
4   1.645550    1.184332      0.0    -0.421824  2.402210 -0.002841  0.157318   

   NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  \
0       0.828777   0.645883       -1.030123         0.264326   
1       0.828777  -1.548267       -1.030123        -0.357358   
2      -0.929809   0.645883        0.970757         1.161131   
3      -0.929809  -1.548267        0.970757        -1.636175   
4       2.587364  -1.548267       -1.030123         1.607949   

   BalanceSalaryRatio  EngagementScore  Geography_Germany  Geography_Spain  \
0           -0.036998        -0.912493          -0.57636

In [34]:
from sklearn.model_selection import train_test_split

# 80-20 split using processed features
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# verify shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (7199, 20)
X_test: (1800, 20)
y_train: (7199,)
y_test: (1800,)


In [35]:
# balance the dataset (if you think it is necessary)
# initialize SMOTE
smote = SMOTE(random_state=42)

# balance only training dataset
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train,
    y_train
)

# class distribution check
print("Before SMOTE:")
print(Counter(y_train))

print("\nAfter SMOTE:")
print(Counter(y_train_balanced))

Before SMOTE:
Counter({0.0: 5728, 1.0: 1471})

After SMOTE:
Counter({0.0: 5728, 1.0: 5728})


In [36]:
# Train a decision tree model using the `train sample`
from sklearn.ensemble import RandomForestClassifier

# initialize random forest model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

# train model
model.fit(X_train_balanced, y_train_balanced)

print("Random Forest Training Complete") # Best performance so far

Random Forest Training Complete


In [37]:
# Check the accuracy of the model on the `test sample`
# make predictions on untouched test set

# predictions
y_pred = model.predict(X_test)

# probabilities
y_prob = model.predict_proba(X_test)[:, 1]

print("Prediction Complete")

Prediction Complete


In [38]:
# Analyze performance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8327777777777777
Precision: 0.5831265508684863
Recall: 0.6385869565217391
F1 Score: 0.6095979247730221
ROC AUC: 0.8431579426767063

Classification Report
              precision    recall  f1-score   support

         0.0       0.90      0.88      0.89      1432
         1.0       0.58      0.64      0.61       368

    accuracy                           0.83      1800
   macro avg       0.74      0.76      0.75      1800
weighted avg       0.84      0.83      0.84      1800


Confusion Matrix
[[1264  168]
 [ 133  235]]


In [39]:
# import packages for mlfow
import mlflow
from mlflow.models import infer_signature
print(mlflow.__version__)

3.12.0


In [40]:
# Set our tracking server uri for logging on port 8080
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")
print(mlflow.get_tracking_uri())

http://127.0.0.1:8080


In [ ]:
# mlflow server \
# --host 127.0.0.1 \
# --port 8080 \
# --backend-store-uri sqlite:///mlflow.db \
# --default-artifact-root ./mlruns

In [41]:
# Create a new MLflow Experiment called `Practice Experiment - {Your Name}`
mlflow.set_experiment("Practice Experiment - Mark Joshua Ponciano Halili")

<Experiment: artifact_location='/Users/mjhalili/Desktop/EADA_Files/ML_Operations/mlops-and-systems-design/session_3/mlruns/1', creation_time=1779626670093, experiment_id='1', last_update_time=1779626670093, lifecycle_stage='active', name='Practice Experiment - Mark Joshua Ponciano Halili', tags={}, trace_location=None, workspace='default'>

In [42]:
# define parameters manually
params = {
    "model_type": "RandomForestClassifier",
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": 42
}

In [43]:
f1 = f1_score(y_test, y_pred)
auc_halili = roc_auc_score(y_test, y_prob)

In [44]:
with mlflow.start_run():

    mlflow.log_params(params)

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("recall", recall_score(y_test, y_pred))
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("AUC_ROC", auc_halili)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Session 2 Class Exercise", "Data Preparation, Model Training and Prediction")

🏃 View run fun-crab-980 at: http://127.0.0.1:8080/#/experiments/1/runs/12b0861953274052bb606a7f982fc83a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [45]:
# Start an MLflow run

with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1)

    # Log the AUC ROC
    mlflow.log_metric("AUC ROC", auc_halili)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Session 2 Class Exercise", "Data Preparation, Model Training and Prediction")

    # infer model signature
    signature = infer_signature(X_train, model.predict(X_train))

    # log the model
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="random_forest_model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )



2026/05/24 21:38:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/24 21:38:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run valuable-horse-195 at: http://127.0.0.1:8080/#/experiments/1/runs/59977cf7e84e4bc1a65aed37db1c72a7
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [46]:
from sklearn.preprocessing import OneHotEncoder
import joblib

In [47]:
PATH = "./artifacts/"
import os
os.makedirs(PATH, exist_ok=True)

In [48]:
# Implement transformation logic as in session 2

categorical_cols = ['Geography', 'Gender', 'AgeGroup']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(clean_df[categorical_cols])
joblib.dump(encoder, f"{PATH}one_hot_encoder.pkl")

['./artifacts/one_hot_encoder.pkl']

In [49]:
# Perform another experiment if you don't have the ones from session 2. Otherwise, this part can be skipped

# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [50]:
# another experiment
import mlflow
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from mlflow.models import infer_signature

In [51]:


# Define experiment name
mlflow.set_experiment("Decision Tree Experiment - Quick Run")

# Split dataset (using your existing df)
X = df.drop(["Exited", "Surname"], axis=1)
X = pd.get_dummies(X)

y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Simple model
model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions + metric
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# MLflow logging
with mlflow.start_run():
    mlflow.log_param("model_type", "DecisionTreeClassifier")
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_split", 4)
    mlflow.log_param("min_samples_leaf", 2)

    mlflow.log_metric("accuracy", acc)

    signature = infer_signature(X_train, model.predict(X_train))

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="decision_tree_model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name="quick-churn-model"
    )

print("MLflow experiment completed successfully.")

2026/05/24 21:38:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/24 21:38:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'quick-churn-model' already exists. Creating a new version of this model...
2026/05/24 21:38:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: quick-churn-model, version 2


🏃 View run gifted-moth-769 at: http://127.0.0.1:8080/#/experiments/2/runs/defb162937414f6e9dde6891ebdf5716
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/2
MLflow experiment completed successfully.


Created version '2' of model 'quick-churn-model'.


In [121]:
import joblib

# save training schema
feature_columns = X_train.columns.tolist()

joblib.dump(feature_columns, "./artifacts/feature_columns.pkl")

['./artifacts/feature_columns.pkl']

In [122]:
import pandas as pd
import joblib

# load training schema
FEATURE_COLUMNS = joblib.load("./artifacts/feature_columns.pkl")



def prepare_input(df: pd.DataFrame):
    df = df.copy()

    # remove columns not used by the model
    drop_cols = [
        "RowNumber",
        "CustomerId",
        "Surname",
        "Exited"
    ]

    df = df.drop(
        columns=[c for c in drop_cols if c in df.columns],
        errors="ignore"
    )

    # feature engineering
    df["BalanceSalaryRatio"] = (
        df["Balance"] /
        (df["EstimatedSalary"] + 1)
    )

    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[0, 25, 35, 50, 65, 100],
        labels=[
            "18-25",
            "26-35",
            "36-50",
            "51-65",
            "65+"
        ]
    )

    df["EngagementScore"] = (
        df["NumOfProducts"] *
        df["IsActiveMember"]
    )

    # one hot encoding
    df = pd.get_dummies(
        df,
        columns=["Geography", "Gender", "AgeGroup"],
        drop_first=True
    )

    # align with training schema
    df = df.reindex(
        columns=FEATURE_COLUMNS,
        fill_value=0
    )

    return df

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [115]:
import os
print(os.getcwd())

/Users/mjhalili/Desktop/EADA_Files/ML_Operations/mlops-and-systems-design/session_3/act/Class_Exercise


In [116]:
import pandas as pd

def prepare_input(df):
    df = df.copy()

    # HARD FIX: always remove non-model columns
    drop_cols = ["RowNumber", "CustomerId", "Surname", "Exited"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    # feature engineering
    df["BalanceSalaryRatio"] = df["Balance"] / (df["EstimatedSalary"] + 1)

    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[0, 25, 35, 50, 65, 100],
        labels=["18-25", "26-35", "36-50", "51-65", "65+"]
    )

    df["EngagementScore"] = df["NumOfProducts"] * df["IsActiveMember"]

    # encoding
    df = pd.get_dummies(df, columns=["Geography", "Gender", "AgeGroup"], drop_first=True)

    return df

In [123]:
import mlflow

MODEL_URI = "runs:/e20cac0551944574afd2c7555e82e132/random_forest_model"

model = mlflow.sklearn.load_model(MODEL_URI)

##### Batch Inference

In [124]:
def batch_inference(model, df: pd.DataFrame):

    original_data = df.copy()

    processed_data = prepare_input(df)

    predictions = model.predict(processed_data)

    probabilities = model.predict_proba(processed_data)[:, 1]

    results = original_data.copy()

    results["prediction"] = predictions

    results["churn_probability"] = probabilities

    return results

In [125]:
# load validation dataset
validation_df = pd.read_csv("./Churn_Modelling_val.csv")

# run inference
batch_results = batch_inference(
    model=model,
    df=validation_df
)

batch_results.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,prediction,churn_probability
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.00,2,1.0,1.0,74900.90,0,0.0,0.006289
1,4685,15736963,Herring,623,France,Male,43.0,1,0.00,2,1.0,1.0,146379.30,0,0.0,0.000000
2,1732,15721730,Amechi,601,Spain,Female,44.0,4,0.00,2,1.0,0.0,58561.31,0,0.0,0.000000
3,4743,15762134,Liang,506,Germany,Male,59.0,8,119152.10,2,1.0,1.0,170679.74,0,0.0,0.000000
4,4522,15648898,Chuang,560,Spain,Female,27.0,7,124995.98,1,1.0,1.0,114669.79,0,1.0,1.000000


In [126]:
print(batch_results.shape)
print(batch_results["Exited"].value_counts())
print(batch_results["prediction"].value_counts())

(1001, 16)
Exited
0    803
1    198
Name: count, dtype: int64
prediction
0.0    649
1.0    352
Name: count, dtype: int64


In [146]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    classification_report
)

import pandas as pd

# confusion matrix
cm = confusion_matrix(
    batch_results["Exited"],
    batch_results["prediction"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_No", "Actual_Yes"],
    columns=["Pred_No", "Pred_Yes"]
)

print("Confusion Matrix:")
display(cm_df)

# accuracy
accuracy = accuracy_score(
    batch_results["Exited"],
    batch_results["prediction"]
)

print(f"Accuracy: {accuracy:.4f}")

# f1 score
f1 = f1_score(
    batch_results["Exited"],
    batch_results["prediction"]
)

print(f"F1 Score: {f1:.4f}")

# optional detailed report
report = classification_report(
    batch_results["Exited"],
    batch_results["prediction"]
)

print("\nClassification Report:")
print(report)

Confusion Matrix:


,Pred_No,Pred_Yes
Actual_No,544,259
Actual_Yes,105,93


Accuracy: 0.6364
F1 Score: 0.3382

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.68      0.75       803
           1       0.26      0.47      0.34       198

    accuracy                           0.64      1001
   macro avg       0.55      0.57      0.54      1001
weighted avg       0.72      0.64      0.67      1001



##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [ ]:
# mlflow models serve -m runs:/e20cac0551944574afd2c7555e82e132/random_forest_model -p 5000 --no-conda

In [147]:

def online_inference(
    record: pd.DataFrame,
    model
):

    processed = prepare_input(record)

    prediction = model.predict(processed)[0]

    probability = model.predict_proba(processed)[0][1]

    return {
        "prediction": int(prediction),
        "probability": float(probability)
    }

In [148]:
sample = validation_df.iloc[[0]]

online_result = online_inference(
    record=sample,
    model=model
)

online_result

{'prediction': 0, 'probability': 0.057971014492753624}

In [149]:
import requests


def get_inference_endpoint(
    host="http://127.0.0.1",
    port=5000
):
    return f"{host}:{port}/invocations"


url = get_inference_endpoint()

In [150]:

def online_inference_pandas(
    url: str,
    input_df: pd.DataFrame
):

    payload = {
        "dataframe_split": {
            "columns": input_df.columns.tolist(),
            "data": input_df.values.tolist()
        }
    }

    response = requests.post(
        url=url,
        headers={"Content-Type": "application/json"},
        json=payload
    )

    return response

In [151]:
sample = validation_df.iloc[[0]]

processed_sample = prepare_input(sample)

response_pandas = online_inference_pandas(
    url=url,
    input_df=processed_sample
)

response_pandas.json()

{'predictions': [0.0]}

In [152]:

def online_inference_json(
    url: str,
    input_json: dict
):

    response = requests.post(
        url=url,
        headers={"Content-Type": "application/json"},
        json=input_json
    )

    return response

In [153]:
sample = validation_df.iloc[[0]]

processed_sample = prepare_input(sample)

input_json = {
    "dataframe_split": {
        "columns": processed_sample.columns.tolist(),
        "data": processed_sample.values.tolist()
    }
}

In [154]:
response_json = online_inference_json(
    url=url,
    input_json=input_json
)

response_json.json()

{'predictions': [0.0]}

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [ ]:
# transform data - if necessary

In [155]:
import requests
import pandas as pd

# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(
    url: str,
    input: pd.DataFrame
):

    payload = {
        "dataframe_split": {
            "columns": input.columns.tolist(),
            "data": input.values.tolist()
        }
    }

    response = requests.post(
        url=url,
        headers={"Content-Type": "application/json"},
        json=payload
    )

    return response

In [156]:
# get sample data
sample = validation_df.iloc[[0]]

# preprocess
processed_sample = prepare_input(sample)

# define endpoint
url = "http://127.0.0.1:5000/invocations"

# call inference
response_pandas = online_inference_pandas(
    url=url,
    input=processed_sample
)

# output
response_pandas.content

b'{"predictions": [0.0]}'

In [157]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(
    url: str,
    input: dict
):

    response = requests.post(
        url=url,
        headers={"Content-Type": "application/json"},
        json=input
    )

    return response

In [158]:
# sample row
sample = validation_df.iloc[[0]]

# preprocess sample
processed_sample = prepare_input(sample)

# define the json as required by MLFlow
input_json = {
    "dataframe_split": {
        "columns": processed_sample.columns.tolist(),
        "data": processed_sample.values.tolist()
    }
}

In [159]:
response_json = online_inference_json(
    url=url,
    input=input_json
)

response_json.content

b'{"predictions": [0.0]}'

In [160]:
import mlflow
import mlflow.sklearn
import joblib

with mlflow.start_run():

    # log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("random_state", 42)

    # log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)

    # log model
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="random_forest_model"
    )

    # save feature schema
    feature_columns = X_train.columns.tolist()

    joblib.dump(
        feature_columns,
        "feature_columns.pkl"
    )

    # log feature schema artifact
    mlflow.log_artifact(
        "feature_columns.pkl"
    )

2026/05/25 14:59:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 14:59:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Python(32885) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


🏃 View run upbeat-crane-343 at: http://127.0.0.1:8080/#/experiments/1/runs/081ab14e12634980805dd28685c85d62
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
